# Renon 3D Gaussian Splatting — Probelauf (4 Standpunkte)

Trainiert 3DGS auf dem vorbereiteten COLMAP-Datensatz (`colmap_renon4.zip`, 20 posierte
Pinhole-Bilder + 300k LiDAR-Initpunkte). Die Posen stammen exakt aus den E57-Blöcken,
nicht aus SfM — `scripts/validate_colmap.py` hat sie gegen die Fotos geprüft.

**Vorher:** Menü **Laufzeit → Laufzeittyp ändern → GPU (T4)**. Dann Zellen der Reihe nach.

Dauer auf T4: ~20–40 Min für 30k Iterationen. Ergebnis ist `renon_gaussians.ply`
(läuft danach per Drag&Drop in [SuperSplat](https://playcanvas.com/supersplat/editor)).

In [ ]:
# 1) Vorflugkontrolle: GPU da? Welche torch/CUDA-Kombination?
#    Die Submodul-Builds in Zelle 2 scheitern fast immer an genau dieser Paarung,
#    deshalb steht sie hier schwarz auf weiss, bevor irgendetwas kompiliert wird.
!nvidia-smi -L
import torch, os
print('torch       :', torch.__version__)
print('CUDA (torch):', torch.version.cuda)
assert torch.cuda.is_available(), 'Keine GPU — Laufzeittyp auf GPU umstellen!'
print('GPU         :', torch.cuda.get_device_name(0))
cap = torch.cuda.get_device_capability(0)
os.environ['TORCH_CUDA_ARCH_LIST'] = f'{cap[0]}.{cap[1]}'   # T4=7.5, L4=8.9, A100=8.0
print('Baue fuer Compute Capability', os.environ['TORCH_CUDA_ARCH_LIST'])

In [ ]:
# 2) 3DGS klonen + CUDA-Submodule bauen (~4-6 Min)
#    DREI Submodule, nicht zwei: fused-ssim ist seit Ende 2024 Pflicht, sonst
#    bricht train.py sofort mit ModuleNotFoundError ab.
%cd /content
!rm -rf gaussian-splatting
!git clone --recursive --depth 1 https://github.com/graphdeco-inria/gaussian-splatting
%cd /content/gaussian-splatting
!pip install -q plyfile opencv-python joblib
!pip install -q --no-build-isolation \
    ./submodules/diff-gaussian-rasterization \
    ./submodules/simple-knn \
    ./submodules/fused-ssim

# Hart pruefen, statt dem stillen 'pip -q' zu glauben
import importlib
for m in ('diff_gaussian_rasterization', 'simple_knn._C', 'fused_ssim'):
    importlib.import_module(m)
    print('OK:', m)

In [ ]:
# 3) Datensatz hochladen: colmap_renon4.zip aus data/Renon/ auswaehlen (15 MB)
from google.colab import files
up = files.upload()
!rm -rf /content/renon && mkdir -p /content/renon
!unzip -q -o colmap_renon4.zip -d /content/renon

# Struktur pruefen — 3DGS erwartet <root>/images und <root>/sparse/0
import os, glob
root = '/content/renon'
if not os.path.isdir(f'{root}/sparse'):           # Zip mit Unterordner entpackt?
    inner = glob.glob(f'{root}/*/sparse')
    if inner:
        root = os.path.dirname(inner[0])
n = len(glob.glob(f'{root}/images/*.jpg'))
print('Datensatz-Wurzel:', root)
print('Bilder          :', n)
print('sparse/0        :', sorted(os.listdir(f'{root}/sparse/0')))
assert n == 20, f'20 Bilder erwartet, {n} gefunden'
DATA = root

In [ ]:
# 4) Training. -r 2 rechnet auf 1024 px (VRAM-schonend), --data_device cpu haelt
#    die Bilder im RAM statt im VRAM — auf der 16-GB-T4 ist beides noetig.
%cd /content/gaussian-splatting
!python train.py -s {DATA} -m /content/output/renon \
    --iterations 30000 -r 2 --data_device cpu

In [ ]:
# 5) Ergebnis sichern + herunterladen
import os
ply = '/content/output/renon/point_cloud/iteration_30000/point_cloud.ply'
assert os.path.isfile(ply), 'Kein Ergebnis — Trainingslog oben pruefen'
print(f'{os.path.getsize(ply)/1e6:.1f} MB Gaussians')
!cp {ply} /content/renon_gaussians.ply
from google.colab import files
files.download('/content/renon_gaussians.ply')

---
## Ausweichweg, falls Zelle 2 nicht baut

Die CUDA-Submodule von graphdeco brechen erfahrungsgemaess, sobald Colab eine neuere
torch/CUDA-Kombination ausrollt. `nerfstudio`/`gsplat` bringt vorkompilierte Wheels mit
und liest dasselbe COLMAP-Verzeichnis — die naechste Zelle ersetzt Zelle 2 **und** 4.

In [ ]:
# ALTERNATIVE zu Zelle 2+4 — nur ausfuehren, wenn der Submodul-Build scheiterte
!pip install -q nerfstudio
!ns-train splatfacto --data {DATA} --max-num-iterations 30000 \
    --viewer.quit-on-train-completion True colmap --downscale-factor 2

# Export der Gaussians ins 3DGS-.ply-Format (SuperSplat-kompatibel)
import glob
cfg = sorted(glob.glob('/content/outputs/**/config.yml', recursive=True))[-1]
print('Config:', cfg)
!ns-export gaussian-splat --load-config {cfg} --output-dir /content/export
!cp /content/export/splat.ply /content/renon_gaussians.ply
from google.colab import files
files.download('/content/renon_gaussians.ply')

## Ergebnis ansehen und bewerten

`renon_gaussians.ply` per Drag&Drop in
**[SuperSplat](https://playcanvas.com/supersplat/editor)** — sofort frei begehbar,
fotorealistisch mit korrekter Parallaxe.

**Worauf beim Probelauf zu achten ist:** die vier Standpunkte liegen nahezu
kollinear auf ~10 m. Quer zu dieser Linie fehlt die Parallaxe, dort sind
Duennstellen und Schlieren zu erwarten — das ist eine Aussage ueber die
Aufnahmegeometrie, nicht ueber 3DGS. Genau daran laesst sich ablesen, wie viele
zusaetzliche Standpunkte ein belastbarer Durchlauf braucht.

Mehr Standpunkte beschaffen und Datensatz neu bauen:

```bash
python scripts/zip_remote.py getnested ... <N>        # weitere Setups laden
python scripts/build_colmap.py "data/Renon/e57/*.e57" data/Renon/colmap \
    --init-points 300000 --per-setup-points 100000
python scripts/validate_colmap.py data/Renon/colmap s001_c01.jpg   # Posen pruefen
```